<a href="https://colab.research.google.com/github/weagan/LAYA-JEV/blob/main/laya_typed_decisions_2nd_stage_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laya Typed-Decisions — second-stage fine-tuning + post-training evaluation

This Colab notebook performs an **experimental second fine-tuning pass** starting from
`convaiinnovations/laya-typed-decisions`.

It deliberately keeps the official `test` split untouched:
- baseline evaluation on the shipped checkpoint
- second-stage fine-tuning on the official `train` split, with a held-out validation/calibration slice
- post-training temperature calibration on held-out data
- final evaluation on the official `test` split
- side-by-side metrics and per-workflow results

**Important:** the shipped `laya-typed-decisions` checkpoint was already trained on the 1,200-case
training split, so this is continued training on data the checkpoint has already seen. It is useful
as an experiment, but any gain must come from the untouched test evaluation; do not assume more
epochs will improve accuracy.

The training recipe follows Laya's published RLCD approach: soft probability targets, proper-scoring
reward, group-mean policy-gradient baseline, and soft cross-entropy guidance.


In [1]:
# 1. Colab environment
%pip install -q -U "laya>=0.3.20" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub scipy pandas scikit-learn tabulate

import os
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, laya, transformers, datasets
print("Laya:", getattr(laya, "__version__", "unknown"))
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))
else:
    raise RuntimeError("This notebook is intended for a Colab GPU runtime. Select Runtime -> Change runtime type -> GPU.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.9/128.9 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.0/360.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is 

In [2]:
# 2. Configuration
from pathlib import Path
import json, math, random, time, gc, numpy as np, pandas as pd

START_MODEL_ID = "convaiinnovations/laya-typed-decisions"
DATASET_ID = "LocalLLaMA/typed-decisions"

WORKDIR = Path("/content/laya_second_stage")
BASE_DIR = WORKDIR / "base_checkpoint"
OUTPUT_DIR = WORKDIR / "laya_second_stage"
ITEMS_PATH = WORKDIR / "train_items.pt"

WORKDIR.mkdir(parents=True, exist_ok=True)

# Conservative defaults for a single Colab GPU.
EPOCHS = 2
MICRO_BATCH = 2
GRAD_ACCUM = 16
LR_ENCODER = 5e-6       # deliberately lower than first-stage fine-tuning
LR_HEAD = 2e-5
WEIGHT_DECAY = 0.01
GROUP_SIZE = 4
SIGMA_START = 0.20
SIGMA_END = 0.08

# 10% is never trained on and is used for post-training calibration.
CALIB_FRACTION = 0.10
SEED = 20261006

print("Output:", OUTPUT_DIR)


Output: /content/laya_second_stage/laya_second_stage


In [3]:
# 3. Download the exact starting checkpoint and dataset
from huggingface_hub import snapshot_download
from datasets import load_dataset
from laya.agent import _fix_tokenizer_config

print("Downloading starting checkpoint...")
snapshot_download(START_MODEL_ID, local_dir=str(BASE_DIR))
_fix_tokenizer_config(str(BASE_DIR))

print("Loading official training split...")
ds_train = load_dataset(DATASET_ID, "all", split="train")
ds_test = load_dataset(DATASET_ID, "all", split="test")

print("Train cases:", len(ds_train))
print("Test cases :", len(ds_test))
print("The test split will not be used for training or hyperparameter selection.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading official training split...


README.md:   0%|          | 0.00/15.3k [00:00<?, ?B/s]

all/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  599kB            

...

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

Train cases: 1200
Test cases : 400
The test split will not be used for training or hyperparameter selection.


In [4]:
# 4. Baseline evaluation of the shipped checkpoint
import laya
from collections import defaultdict

baseline_agent = laya.load(START_MODEL_ID)

def gold_distribution(gold_q, q):
    probs = gold_q["probabilities"]
    t = q["type"]

    if t == "choice":
        keys = list(q.get("criteria", {}).keys())
        p = [float(probs.get(k, 0.0)) for k in keys]
    elif t == "noul":
        # Laya's noul output is P(true).
        p_true = float(probs.get("true", 0.0))
        p = [1.0 - p_true, p_true]
    elif t == "score":
        crit = q.get("criteria", [])
        n = len(crit) if isinstance(crit, list) else len(probs)
        p = [float(probs.get(str(i), 0.0)) for i in range(n)]
    else:
        raise ValueError(f"Unsupported question type: {t}")

    s = sum(p)
    return [x / s for x in p] if s > 0 else [1.0 / len(p)] * len(p)


def _question_schema_key(questions):
    """Stable key for batching rows that have the same question schema."""
    return json.dumps(questions, sort_keys=True, separators=(",", ":"))


def _answer_probabilities(answer, question):
    """Convert every Laya answer type to one probability vector."""
    t = question["type"]

    if t == "choice":
        keys = list(question.get("criteria", {}).keys())
        p = [float(answer["probabilities"].get(k, 0.0)) for k in keys]
        pred_idx = int(np.argmax(p))
        pred_score = np.nan

    elif t == "noul":
        p_true = float(answer["noul"])
        p = [1.0 - p_true, p_true]
        pred_idx = int(p_true >= 0.5)
        pred_score = np.nan

    elif t == "score":
        crit = question.get("criteria", [])
        n = len(crit)
        p = [float(answer["probabilities"].get(str(i), 0.0)) for i in range(n)]
        pred_idx = int(np.argmax(p))
        pred_score = float(answer["score"])

    else:
        raise ValueError(f"Unsupported Laya answer type: {t}")

    s = sum(p)
    if s > 0:
        p = [x / s for x in p]
    return p, pred_idx, pred_score


def metric_rows(agent, ds, label):
    """
    Evaluate without assuming that all benchmark rows share one question schema.

    The typed-decisions benchmark contains four workflow-specific schemas. Laya's
    predict_batch() is intended for requests sharing a question schema, so we
    group rows by their exact questions JSON and batch each group separately.
    """
    prepared = []
    groups = defaultdict(list)

    for row_idx, row in enumerate(ds):
        state = json.loads(row["state"])
        questions = json.loads(row["questions"])
        gold = json.loads(row["gold"])

        item = {
            "row_idx": row_idx,
            "state": state,
            "questions": questions,
            "gold": gold,
            "workflow": row["workflow"],
        }
        prepared.append(item)
        groups[_question_schema_key(questions)].append(item)

    print(f"Evaluating {len(prepared)} cases in {len(groups)} question-schema groups.")

    predictions = {}

    for group_num, group_items in enumerate(groups.values(), start=1):
        shared_questions = group_items[0]["questions"]
        states = [x["state"] for x in group_items]

        # All rows in this group have the exact same question schema.
        results = agent.predict_batch(
            states,
            shared_questions,
            batch_size=8
        )

        for item, result in zip(group_items, results):
            predictions[item["row_idx"]] = result

        print(
            f"  group {group_num}/{len(groups)}: "
            f"{len(group_items)} cases, "
            f"questions={list(shared_questions.keys())}"
        )

    rows = []

    for item in prepared:
        res = predictions[item["row_idx"]]
        questions = item["questions"]
        gold = item["gold"]

        if "answers" not in res:
            raise RuntimeError(
                f"Laya prediction has no 'answers' field. "
                f"Returned keys: {list(res.keys())}"
            )

        answers = res["answers"]

        # Helpful diagnostic if a future Laya version changes its response shape.
        missing = [qid for qid in questions if qid not in answers]
        if missing:
            raise RuntimeError(
                f"Prediction is missing question ids {missing}. "
                f"Question ids={list(questions)}, answer ids={list(answers)}"
            )

        for qid, q in questions.items():
            g = gold[qid]
            gp = gold_distribution(g, q)
            a = answers[qid]

            pp, pred_idx, pred_score = _answer_probabilities(a, q)
            gold_idx = int(np.argmax(gp))
            pred_prob_gold = pp[gold_idx]

            brier = sum(
                (pp[i] - gp[i]) ** 2
                for i in range(len(gp))
            )

            kl = sum(
                gp[i] * math.log(
                    max(gp[i], 1e-12) /
                    max(pp[i], 1e-12)
                )
                for i in range(len(gp))
            )

            rows.append({
                "model": label,
                "workflow": item["workflow"],
                "qid": qid,
                "type": q["type"],
                "correct": int(pred_idx == gold_idx),
                "soft_acc": pred_prob_gold,
                "brier": brier,
                "kl": kl,
                "confidence": max(pp),
                "gold_score": (
                    float(sum(i * gp[i] for i in range(len(gp))))
                    if q["type"] == "score" else np.nan
                ),
                "pred_score": (
                    pred_score if q["type"] == "score" else np.nan
                ),
                "n_levels": len(gp)
            })

    return pd.DataFrame(rows)


baseline_df = metric_rows(baseline_agent, ds_test, "shipped")

print("Baseline decisions:", len(baseline_df))
print(
    baseline_df.groupby("type")[["correct", "soft_acc", "brier", "kl"]]
    .mean()
    .round(4)
)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/agent.py:2125: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


Evaluating 400 cases in 4 question-schema groups.
  group 1/4: 100 cases, questions=['action', 'needs_review', 'outcome', 'risk', 'urgency']
  group 2/4: 100 cases, questions=['action', 'category', 'churn_risk', 'needs_human', 'urgency']
  group 3/4: 100 cases, questions=['discrepancy_severity', 'disposition', 'duplicate', 'matches_order', 'urgency']
  group 4/4: 100 cases, questions=['credential_compromise', 'disposition', 'severity', 'true_positive', 'urgency']
Baseline decisions: 2000
        correct  soft_acc   brier      kl
type                                     
choice   0.7333    0.4460  0.0821  0.1736
noul     0.8550    0.6494  0.0494  0.0705
score    0.7250    0.4862  0.0550  0.1095


In [5]:
# 4b. Optional response-shape diagnostic
# Run this if Laya or the benchmark schema changes in a future package version.
_sample_state = json.loads(ds_test[0]["state"])
_sample_questions = json.loads(ds_test[0]["questions"])
_sample_result = baseline_agent.predict(_sample_state, _sample_questions)
print("Question IDs :", list(_sample_questions.keys()))
print("Answer IDs   :", list(_sample_result["answers"].keys()))
for _qid, _ans in _sample_result["answers"].items():
    print(_qid, "->", _ans)


Question IDs : ['action', 'needs_review', 'outcome', 'risk', 'urgency']
Answer IDs   : ['action', 'needs_review', 'outcome', 'risk', 'urgency']
action -> {'type': 'choice', 'choice': 'continue', 'probabilities': {'continue': 0.4322, 'human_review': 0.1981, 'observe': 0.3341, 'stop': 0.0357}, 'confidence': 0.1571, 'answer_confidence': 0.4322, 'action': {'act_probability': 1.0}}
needs_review -> {'type': 'noul', 'noul': 0.2736, 'confidence': 0.7264, 'answer_confidence': 0.7264, 'action': {'act_probability': 1.0}}
outcome -> {'type': 'choice', 'choice': 'success', 'probabilities': {'failure': 0.1204, 'harmful': 0.0433, 'partial': 0.2863, 'success': 0.55}, 'confidence': 0.2226, 'answer_confidence': 0.55, 'action': {'act_probability': 1.0}}
risk -> {'type': 'score', 'score': 0.6636, 'legend': {'0': 'Benign: read-only or clearly safe actions.', '1': 'Low: routine writes within scope.', '2': 'Moderate: irreversible or out-of-scope actions.', '3': 'High: destructive, security-relevant, or polic

In [6]:
# 5. Official-style aggregate metrics
def ece_binary_or_multiclass(df, bins=15):
    if len(df) == 0:
        return np.nan
    conf = df["confidence"].to_numpy(float)
    corr = df["correct"].to_numpy(float)
    edges = np.linspace(0, 1, bins+1)
    out = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
        if m.any():
            out += m.mean() * abs(conf[m].mean() - corr[m].mean())
    return float(out)

def summarize(df):
    score_mask = df["type"].eq("score")
    out = {
        "accuracy": df["correct"].mean(),
        "soft_accuracy": df["soft_acc"].mean(),
        "brier": df["brier"].mean(),
        "kl_from_gold": df["kl"].mean(),
        "ece": ece_binary_or_multiclass(df),
        "score_mae": (df.loc[score_mask, "pred_score"] - df.loc[score_mask, "gold_score"]).abs().mean()
                      if score_mask.any() else np.nan,
        "within_1_level": (
            (df.loc[score_mask, "pred_score"] - df.loc[score_mask, "gold_score"]).abs() <= 1
        ).mean() if score_mask.any() else np.nan
    }
    return pd.Series(out)

print("BASELINE")
display(summarize(baseline_df).round(4))
display(baseline_df.groupby("workflow").apply(summarize, include_groups=False).round(4))


BASELINE


accuracy          0.7665
soft_accuracy     0.5231
brier             0.0615
kl_from_gold      0.1170
ece               0.2137
score_mae         0.2424
within_1_level    0.9950
dtype: float64

,accuracy,soft_accuracy,brier,kl_from_gold,ece,score_mae,within_1_level
workflow,,,,,,,
agent_trace_observability,0.722,0.4659,0.0569,0.1001,0.2213,0.2101,0.995
customer_service,0.760,0.5196,0.0790,0.1564,0.2088,0.2198,0.995
invoice_processing,0.814,0.6176,0.0640,0.1279,0.1749,0.2919,0.995
security_incidents,0.770,0.4893,0.0460,0.0837,0.2529,0.2478,0.995


## 6. Build soft-target training items

The benchmark provides full gold probability distributions, not only hard labels. We retain those
soft targets so the second-stage training does not collapse Laya into ordinary hard-label
classification. The latent `factors` field is intentionally ignored. citeturn1search0turn1search6


In [7]:
# 6. Preprocess the official TRAIN split
from transformers import AutoTokenizer
from laya.common import build_sequence, render_options, QTYPES

tok = AutoTokenizer.from_pretrained(BASE_DIR / "tokenizer")
with open(BASE_DIR / "rl_agent_config.json") as f:
    cfg = json.load(f)

cfg["max_len"] = 1024
cfg["head_max_len"] = 256

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    target = gold_distribution(gold_q, q)
    label = int(np.argmax(target))
    seq, markers = build_sequence(
        tok, state,
        {"t": t, "ins": q["instructions"], "crit": crit},
        cfg["max_len"], cfg["head_max_len"]
    )
    k = len(render_options({"t": t, "crit": crit}))
    if len(markers) != k:
        return None
    return {
        "ids": seq, "markers": markers, "qtype": QTYPES[t],
        "target": target, "label": label
    }

items = []
for row in ds_train:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    for qid, q in questions.items():
        if qid in gold:
            item = build_training_item(state, q, gold[qid])
            if item is not None:
                items.append(item)

print("Training decisions:", len(items))
torch.save(items, ITEMS_PATH)


Training decisions: 6000


In [8]:
# 7. Deterministic train / calibration split
# The calibration slice is withheld BEFORE training and is never optimized on.
order = list(range(len(items)))
random.Random(SEED).shuffle(order)
n_calib = min(max(100, int(len(items)*CALIB_FRACTION)), len(items)//5)

calib_items = [items[i] for i in sorted(order[:n_calib])]
train_items = [items[i] for i in sorted(order[n_calib:])]

print("Train decisions:", len(train_items))
print("Held-out calibration decisions:", len(calib_items))


Train decisions: 5400
Held-out calibration decisions: 600


In [9]:
# 8. Single-GPU second-stage RLCD fine-tuning
from safetensors.torch import load_file, save_file
from laya.common import build_model, proper_reward

device = torch.device("cuda")
model = build_model(cfg, encoder_dir=BASE_DIR / "encoder")
model.load_state_dict(load_file(str(BASE_DIR / "model.safetensors")), strict=True)
model.float()

try:
    model.encoder.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
except TypeError:
    model.encoder.gradient_checkpointing_enable()
model.head_checkpointing = True
model.to(device).train()

def collate(items):
    n = len(items)
    L = max(len(x["ids"]) for x in items)
    K = max(len(x["markers"]) for x in items)
    ids = torch.full((n,L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((n,L), dtype=torch.long)
    pos = torch.zeros((n,K), dtype=torch.long)
    mask = torch.zeros((n,K), dtype=torch.bool)
    target = torch.zeros((n,K), dtype=torch.float32)
    for i,it in enumerate(items):
        ids[i,:len(it["ids"])] = torch.tensor(it["ids"])
        att[i,:len(it["ids"])] = 1
        pos[i,:len(it["markers"])] = torch.tensor(it["markers"])
        mask[i,:len(it["markers"])] = True
        target[i,:len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    qtype = torch.tensor([it["qtype"] for it in items], dtype=torch.long)
    return ids, att, pos, mask, target, qtype

enc_params = [p for n,p in model.named_parameters() if "encoder." in n]
head_params = [p for n,p in model.named_parameters() if "encoder." not in n]
optimizer = torch.optim.AdamW([
    {"params": enc_params, "lr": LR_ENCODER},
    {"params": head_params, "lr": LR_HEAD}
], weight_decay=WEIGHT_DECAY)

updates = max(1, math.ceil(len(train_items)/MICRO_BATCH/GRAD_ACCUM)*EPOCHS)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=updates, eta_min=min(LR_ENCODER, LR_HEAD)*0.1
)
scaler = torch.amp.GradScaler("cuda")

history = []
for epoch in range(EPOCHS):
    random.Random(SEED + epoch).shuffle(train_items)
    optimizer.zero_grad(set_to_none=True)
    total = 0.0
    steps = 0
    sigma = SIGMA_START + (SIGMA_END-SIGMA_START)*epoch/max(1,EPOCHS-1)

    for start in range(0, len(train_items), MICRO_BATCH):
        chunk = train_items[start:start+MICRO_BATCH]
        ids,att,pos,mask,target,qtype = collate(chunk)
        ids,att,pos,mask,target,qtype = [x.to(device) for x in (ids,att,pos,mask,target,qtype)]

        with torch.autocast("cuda", dtype=torch.float16):
            logits, activation = model(ids,att,pos,mask,qtype)
        logits = logits.float()

        k = mask.sum(-1, keepdim=True).float()
        eps = torch.randn((GROUP_SIZE,)+logits.shape, device=device) * sigma * mask
        eps = (eps - eps.sum(-1, keepdim=True)/k) * mask
        noisy = logits.detach().unsqueeze(0) + eps
        probs = torch.softmax(noisy.masked_fill(~mask, -1e4), -1)

        with torch.no_grad():
            reward = proper_reward(
                probs, target.unsqueeze(0), qtype, mask,
                w_sph=0.75, w_rps=1.0
            )
            adv = reward - reward.mean(0, keepdim=True)
            adv = adv / (adv.std() + 1e-6)

        logp = -(((noisy-logits.unsqueeze(0))**2)*mask).sum(-1)/(2*sigma**2)
        loss_rl = -(adv*logp).mean()
        loss_ce = -(target*torch.log_softmax(
            logits.masked_fill(~mask, -1e4), -1
        )).sum(-1).mean()

        loss = (loss_rl + loss_ce + 0.0*activation.sum()) / GRAD_ACCUM
        scaler.scale(loss).backward()
        steps += 1
        total += float(loss.item()*GRAD_ACCUM)

        if steps % GRAD_ACCUM == 0 or start+MICRO_BATCH >= len(train_items):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if steps % 100 == 0:
            print(f"epoch {epoch+1}/{EPOCHS} step {steps} loss={loss.item()*GRAD_ACCUM:.4f} reward={reward.mean().item():.4f}")

    avg = total/max(1,steps)
    history.append({"epoch":epoch+1,"loss":avg,"sigma":sigma})
    print(f"Epoch {epoch+1}: avg loss={avg:.4f}")

pd.DataFrame(history)


/tmp/ipykernel_491/529751895.py:97: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


epoch 1/2 step 100 loss=0.0791 reward=0.3394
epoch 1/2 step 200 loss=1.9329 reward=-0.7570
epoch 1/2 step 300 loss=0.7151 reward=-0.0634
epoch 1/2 step 400 loss=0.8611 reward=-0.8105
epoch 1/2 step 500 loss=0.5874 reward=-0.2801
epoch 1/2 step 600 loss=-0.4957 reward=-0.4971
epoch 1/2 step 700 loss=1.6832 reward=-0.3514
epoch 1/2 step 800 loss=0.1848 reward=0.1242
epoch 1/2 step 900 loss=-0.5011 reward=0.1550
epoch 1/2 step 1000 loss=0.4904 reward=-0.1785
epoch 1/2 step 1100 loss=0.2463 reward=-0.1019
epoch 1/2 step 1200 loss=0.2671 reward=-0.0875
epoch 1/2 step 1300 loss=0.6308 reward=-0.6066
epoch 1/2 step 1400 loss=0.1646 reward=0.0004
epoch 1/2 step 1500 loss=0.5901 reward=0.0917
epoch 1/2 step 1600 loss=0.8427 reward=-0.0864
epoch 1/2 step 1700 loss=0.3696 reward=0.1263
epoch 1/2 step 1800 loss=-0.2481 reward=0.1987
epoch 1/2 step 1900 loss=-0.2986 reward=-0.0890
epoch 1/2 step 2000 loss=0.1559 reward=-0.1850
epoch 1/2 step 2100 loss=0.7758 reward=-0.4764
epoch 1/2 step 2200 loss=

,epoch,loss,sigma
0,1,0.589656,0.20
1,2,0.695432,0.08


In [10]:
# 9. Post-training calibration on the held-out calibration slice
model.eval()

def fit_temperature(samples):
    if len(samples) < 10:
        return 1.0
    kmax = max(len(z) for z,_ in samples)
    Z = torch.full((len(samples),kmax), -1e4)
    T = torch.zeros((len(samples),kmax))
    for i,(z,t) in enumerate(samples):
        Z[i,:len(z)] = torch.tensor(z)
        T[i,:len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T*torch.log_softmax(Z/log_t.exp(),-1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(),0.1,10.0).item())

samples = [[] for _ in range(3)]
with torch.no_grad():
    for start in range(0,len(calib_items),MICRO_BATCH):
        chunk = calib_items[start:start+MICRO_BATCH]
        ids,att,pos,mask,target,qtype = collate(chunk)
        logits,_ = model(ids.to(device),att.to(device),pos.to(device),mask.to(device),qtype.to(device))
        logits = logits.float().cpu()
        for i,it in enumerate(chunk):
            k = len(it["markers"])
            samples[it["qtype"]].append((logits[i,:k].numpy(), it["target"]))

temperatures = [fit_temperature(x) if x else 1.0 for x in samples]
print("Fitted temperatures [choice, score, noul]:", [round(x,4) for x in temperatures])

# Persist the calibrated checkpoint.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
weights = {k:v.detach().half().cpu().contiguous() for k,v in model.state_dict().items()}
save_file(weights, str(OUTPUT_DIR/"model.safetensors"))
model.encoder.config.save_pretrained(OUTPUT_DIR/"encoder")
tok.save_pretrained(OUTPUT_DIR/"tokenizer")

cfg_out = dict(cfg)
cfg_out.update({
    "fine_tuned": True,
    "model_name": "laya-typed-decisions-second-stage",
    "temperature": [float(x) for x in temperatures],
    "second_stage_epochs": EPOCHS,
    "second_stage_seed": SEED
})
cfg_out.pop("temperature_by_options", None)
with open(OUTPUT_DIR/"rl_agent_config.json","w") as f:
    json.dump(cfg_out,f,indent=2)

print("Saved:", OUTPUT_DIR)


Fitted temperatures [choice, score, noul]: [1.052, 1.1022, 1.0366]
Saved: /content/laya_second_stage/laya_second_stage


In [11]:
# 10. Load the exported checkpoint through the public Laya API
del model, optimizer, scheduler, scaler
gc.collect()
torch.cuda.empty_cache()

tuned_agent = laya.load(str(OUTPUT_DIR))
print("Second-stage checkpoint loaded successfully.")


Second-stage checkpoint loaded successfully.


In [12]:
# 11. POST-TRAINING EVALUATION on the untouched official TEST split
tuned_df = metric_rows(tuned_agent, ds_test, "second_stage")

baseline_summary = summarize(baseline_df).rename("shipped")
tuned_summary = summarize(tuned_df).rename("second_stage")
comparison = pd.concat([baseline_summary, tuned_summary], axis=1)
comparison["delta"] = comparison["second_stage"] - comparison["shipped"]

print("Overall comparison")
display(comparison.round(4))

print("Per-workflow comparison")
base_w = baseline_df.groupby("workflow").apply(summarize, include_groups=False)
tune_w = tuned_df.groupby("workflow").apply(summarize, include_groups=False)
workflow_compare = pd.concat(
    {"shipped":base_w, "second_stage":tune_w}, axis=1
)
display(workflow_compare.round(4))


Evaluating 400 cases in 4 question-schema groups.
  group 1/4: 100 cases, questions=['action', 'needs_review', 'outcome', 'risk', 'urgency']
  group 2/4: 100 cases, questions=['action', 'category', 'churn_risk', 'needs_human', 'urgency']
  group 3/4: 100 cases, questions=['discrepancy_severity', 'disposition', 'duplicate', 'matches_order', 'urgency']
  group 4/4: 100 cases, questions=['credential_compromise', 'disposition', 'severity', 'true_positive', 'urgency']
Overall comparison


,shipped,second_stage,delta
accuracy,0.7665,0.7850,0.0185
soft_accuracy,0.5231,0.5912,0.0681
brier,0.0615,0.0510,-0.0104
kl_from_gold,0.1170,0.0938,-0.0232
ece,0.2137,0.1492,-0.0645
score_mae,0.2424,0.2214,-0.0210
within_1_level,0.9950,0.9938,-0.0012


Per-workflow comparison


shipped                                             \
                          accuracy soft_accuracy   brier kl_from_gold     ece   
workflow                                                                        
agent_trace_observability    0.722        0.4659  0.0569       0.1001  0.2213   
customer_service             0.760        0.5196  0.0790       0.1564  0.2088   
invoice_processing           0.814        0.6176  0.0640       0.1279  0.1749   
security_incidents           0.770        0.4893  0.0460       0.0837  0.2529   

                                                   second_stage                \
                          score_mae within_1_level     accuracy soft_accuracy   
workflow                                                                        
agent_trace_observability    0.2101          0.995        0.742        0.5209   
customer_service             0.2198          0.995        0.770        0.5926   
invoice_processing           0.2919          0.995        0.852        0.7069   
security_incidents           0.2478          0.995        0.776        0.5445   

                                                                  \
                            brier kl_from_gold     ece score_mae   
workflow                                                           
agent_trace_observability  0.0509       0.0876  0.1681    0.2110   
customer_service           0.0660       0.1250  0.1219    0.2035   
invoice_processing         0.0471       0.0905  0.1143    0.2276   
security_incidents         0.0401       0.0720  0.1938    0.2433   

                                          
                          within_1_level  
workflow                                  
agent_trace_observability          1.000  
customer_service                   0.995  
invoice_processing                 0.985  
security_incidents                 0.995

In [13]:
# 12. Per-question-type comparison
base_t = baseline_df.groupby("type")[["correct","soft_acc","brier","kl"]].mean()
tune_t = tuned_df.groupby("type")[["correct","soft_acc","brier","kl"]].mean()
type_compare = pd.concat({"shipped":base_t, "second_stage":tune_t}, axis=1)
display(type_compare.round(4))


shipped                          second_stage                         
       correct soft_acc   brier      kl      correct soft_acc   brier      kl
type                                                                         
choice  0.7333   0.4460  0.0821  0.1736       0.7450   0.5545  0.0676  0.1312
noul    0.8550   0.6494  0.0494  0.0705       0.8567   0.7250  0.0351  0.0467
score   0.7250   0.4862  0.0550  0.1095       0.7612   0.5184  0.0505  0.1010

In [14]:
# 13. Save a machine-readable post-training report
report = {
    "starting_checkpoint": START_MODEL_ID,
    "dataset": DATASET_ID,
    "test_cases": len(ds_test),
    "test_decisions": len(tuned_df),
    "training_decisions": len(train_items),
    "calibration_decisions": len(calib_items),
    "epochs": EPOCHS,
    "seed": SEED,
    "temperatures_choice_score_noul": temperatures,
    "shipped": summarize(baseline_df).to_dict(),
    "second_stage": summarize(tuned_df).to_dict(),
    "delta": (summarize(tuned_df)-summarize(baseline_df)).to_dict()
}
with open(OUTPUT_DIR/"post_training_report.json","w") as f:
    json.dump(report,f,indent=2)

baseline_df.to_csv(OUTPUT_DIR/"baseline_test_decisions.csv", index=False)
tuned_df.to_csv(OUTPUT_DIR/"second_stage_test_decisions.csv", index=False)

print(json.dumps(report, indent=2))


{
  "starting_checkpoint": "convaiinnovations/laya-typed-decisions",
  "dataset": "LocalLLaMA/typed-decisions",
  "test_cases": 400,
  "test_decisions": 2000,
  "training_decisions": 5400,
  "calibration_decisions": 600,
  "epochs": 2,
  "seed": 20261006,
  "temperatures_choice_score_noul": [
    1.052011489868164,
    1.102168083190918,
    1.0365537405014038
  ],
  "shipped": {
    "accuracy": 0.7665,
    "soft_accuracy": 0.523095934849044,
    "brier": 0.06146414572118184,
    "kl_from_gold": 0.11702894158043026,
    "ece": 0.21373947718416542,
    "score_mae": 0.24240676285063933,
    "within_1_level": 0.995
  },
  "second_stage": {
    "accuracy": 0.785,
    "soft_accuracy": 0.591235642494467,
    "brier": 0.05102812166886932,
    "kl_from_gold": 0.0937811323056752,
    "ece": 0.1492127765323426,
    "score_mae": 0.2213600367695674,
    "within_1_level": 0.99375
  },
  "delta": {
    "accuracy": 0.018500000000000072,
    "soft_accuracy": 0.06813970764542299,
    "brier": -0.010436

## How to interpret the result

A successful run should **not** be judged by training loss alone.

For this benchmark, look especially at:
- **Accuracy** — hard decision correctness.
- **Soft accuracy** — probability assigned to the gold decision.
- **Brier** and **KL** — quality of the probability distribution.
- **ECE** — calibration.
- **Score MAE / within-one-level** — ordinal score quality.
- **Per-workflow metrics** — to detect gains that are really regressions in one workflow.

The published checkpoint reports about **0.766 accuracy, 0.062 Brier, 0.213 ECE and 0.242 score MAE** on the official test split. Its model card also explicitly warns that it remains over-confident and that calibration should be refit on held-out data. citeturn0search0turn4search0

Because this notebook starts from that already-fine-tuned checkpoint, a second-stage run that does not beat those numbers is not a failure of the notebook; it is evidence that the extra training is not helping. In that case, the next improvement should be **new domain data**, stronger validation, or targeted option-order augmentation rather than simply adding epochs.
